In [1]:
# Importar bibliotecas necessárias
import os
from dotenv import load_dotenv
from minio import Minio
from minio.error import S3Error
import pandas as pd

# Carregar variáveis de ambiente do .env
load_dotenv()

# Configurações do MinIO
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# Configurar o cliente MinIO
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

# Nome do bucket
bucket_name = "bronze"

# Criar o bucket se não existir
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

# Verificar se os dados transformados estão disponíveis localmente
if not os.path.exists('../data/bronze/nafld1.parquet') or not os.path.exists('../data/bronze/nafld2.parquet'):
    # Carregar dados brutos
    df_raw1 = pd.read_csv('../data/raw/nafld1.csv')
    df_raw2 = pd.read_csv('../data/raw/nafld2.csv')

    # Realizar transformações necessárias nos dados
    df_bronze1 = df_raw1  # Exemplo: Supondo que bronze1 é uma cópia direta de raw1
    df_bronze2 = df_raw2  # Exemplo: Supondo que bronze2 é uma cópia direta de raw2

    # Salvar dados transformados na camada Bronze
    df_bronze1.to_parquet('../data/bronze/nafld1.parquet')
    df_bronze2.to_parquet('../data/bronze/nafld2.parquet')

# Fazer upload dos arquivos Parquet para o bucket bronze no MinIO
try:
    client.fput_object(
        bucket_name, "nafld1.parquet", "../data/bronze/nafld1.parquet"
    )
    client.fput_object(
        bucket_name, "nafld2.parquet", "../data/bronze/nafld2.parquet"
    )
    print("Upload para o MinIO concluído com sucesso!")
except S3Error as e:
    print(f"Erro ao fazer upload para o MinIO: {e}")

Upload para o MinIO concluído com sucesso!
